# 04 · Model Router — one endpoint, many models

Contoso Outdoors' support inbox gets everything from "what's your return policy?" to genuinely hard trip-planning questions. Picking one model for all of it means either overpaying on the easy ones or under-serving the hard ones. Model Router picks per request.

**Learning objectives**
- Send 25 varied prompts across 5 categories through `model-router`
- See which underlying model actually answers each category (`served_model`)
- Read the cost/latency trade-off the router is making for you
- Know when routing helps and when it doesn't

`~20 minutes`


## 1 · Set up and a helper to compare runs

Same `.env` as every other lab. `ask(model, prompt)` returns the answer plus `served_model` — the one field that matters here, since a router can serve a different model than the one you asked for.


In [1]:
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient


def find_agent_builder() -> Path:
    """Locate foundry/agent-builder from anywhere in the tree (repo root or labs/more)."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "foundry" / "agent-builder" / "src").is_dir():
            return base / "foundry" / "agent-builder"
        if base.name == "agent-builder" and (base / "src").is_dir():
            return base
    raise FileNotFoundError("Run labs/core/00-validate-setup.ipynb first — src/.env not found.")


AB = find_agent_builder()
load_dotenv(AB / "src" / ".env")
project_client = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)
openai_client = project_client.get_openai_client()


def ask(model: str, prompt: str) -> dict:
    """Send one prompt via the Responses API; capture answer, latency, tokens.

    Responses API (not Chat Completions): the newer, recommended shape on Foundry.
    """
    start = time.perf_counter()
    resp = openai_client.responses.create(model=model, input=prompt)
    return {
        "asked_model": model,
        "served_model": resp.model,
        "latency_s": round(time.perf_counter() - start, 2),
        "total_tokens": getattr(resp.usage, "total_tokens", None),
        "answer": resp.output_text,
    }


print("Ready. `ask(model, prompt)` returns served_model, latency, and tokens.")


Ready. `ask(model, prompt)` returns served_model, latency, and tokens.


## 2 · Send 25 prompts across 5 categories through the router

Five categories, five prompts each — from a trivial lookup to a deliberately ambiguous trick question. Every prompt goes to the same `model-router` deployment.

> ❓ **Does `served_model` change across categories, or does the router send everything to the same place?**


In [2]:
PROMPTS = {
    "simple_lookup": [
        "What material is a rainfly usually made of?",
        "In one sentence, what does a bear canister do?",
        "What's the difference between a 2-season and 4-season tent, in one line?",
        "Name one benefit of a merino wool base layer.",
        "What does 'taped seams' mean on a jacket?",
    ],
    "comparison": [
        "Compare a 30L and 50L backpack — which trip length fits each?",
        "Down vs synthetic insulation: one key trade-off for wet climates.",
        "Trekking poles vs no poles for steep descents — quick take.",
        "Freeze-dried meals vs fresh food for a 3-day trip — pros/cons in brief.",
        "Hardshell vs softshell jacket — when would you pick each?",
    ],
    "complex_reasoning": [
        "Plan a 5-day gear list under a $600 budget for shoulder-season weather uncertainty, with trade-off reasoning.",
        "A customer's tent leaked in light rain but passed a hose test at home — walk through likely causes step by step.",
        "Design a returns policy exception process for gear damaged on a first use, balancing customer trust and fraud risk.",
        "Reason through whether a beginner should buy or rent gear for one trip, considering cost-per-use and commitment.",
        "A group of 6 with mixed fitness levels wants one itinerary — reason through pacing and bailout options.",
    ],
    "marketing_copy": [
        "Write a 2-sentence product blurb for a lightweight rain jacket.",
        "Write an email subject line for a fall hiking gear sale.",
        "Write a tagline for a family camping bundle.",
        "Write 3 bullet points selling a 4-season tent's durability.",
        "Write a short Instagram caption for a sunrise summit photo.",
    ],
    "ambiguous_trick": [
        "Is this tent good?",
        "What's the best gear?",
        "Fix my order.",
        "It doesn't work.",
        "How much is it?",
    ],
}

router_runs = []
for category, prompts in PROMPTS.items():
    for prompt in prompts:
        r = ask("model-router", prompt)
        r["category"] = category
        router_runs.append(r)

df = pd.DataFrame(router_runs)
print(f"Ran {len(df)} prompts across {len(PROMPTS)} categories.")
df[["category", "asked_model", "served_model", "latency_s", "total_tokens"]]


Ran 25 prompts across 5 categories.


,category,asked_model,served_model,latency_s,total_tokens
0,simple_lookup,model-router,gpt-5-mini-2025-08-07,8.62,741
1,simple_lookup,model-router,gpt-5-mini-2025-08-07,2.68,326
2,simple_lookup,model-router,gpt-oss-120b,1.92,229
3,simple_lookup,model-router,gpt-oss-120b,1.16,175
4,simple_lookup,model-router,gpt-5-mini-2025-08-07,8.75,1620
5,comparison,model-router,gpt-5.6-luna-2026-07-09,4.48,391
6,comparison,model-router,grok-4-1-fast-reasoning,4.24,423
7,comparison,model-router,gpt-5.6-luna-2026-07-09,3.20,136
8,comparison,model-router,grok-4-1-fast-reasoning,4.56,526
9,comparison,model-router,gpt-5.6-luna-2026-07-09,5.19,445


## 3 · Read the routing pattern

25 individual rows are hard to eyeball. Group by category and see which `served_model` dominates each one, plus the average latency and tokens the router spent there.

> ❓ **Your call:** did the ambiguous/trick category get routed differently than the others? What would you want the router to do with a prompt like "Fix my order"?


In [3]:
summary = df.groupby("category").agg(
    most_common_model=("served_model", lambda s: s.mode().iat[0]),
    distinct_models=("served_model", "nunique"),
    avg_latency_s=("latency_s", "mean"),
    avg_tokens=("total_tokens", "mean"),
).round(2)

summary


,most_common_model,distinct_models,avg_latency_s,avg_tokens
category,,,,
ambiguous_trick,gpt-5-mini-2025-08-07,4,6.21,843.8
comparison,gpt-5.6-luna-2026-07-09,2,4.33,384.2
complex_reasoning,gpt-5-mini-2025-08-07,3,20.28,3168.8
marketing_copy,gpt-oss-120b,2,3.17,391.4
simple_lookup,gpt-5-mini-2025-08-07,2,4.63,618.2


## 4 · One thing to know — routing mode is a deployment setting, not a per-request one

You can't pass "prefer cheaper" or "prefer quality" as a chat parameter. Routing mode (balanced / cost / quality) and which models are eligible are configured **on the deployment itself**, in the Foundry portal or via `az rest` against the deployment resource. Your `model-router` deployment here uses whatever mode it was created with.

That's a provisioning decision for later — this notebook is about understanding *that* routing happens and reading its trade-offs, not about standing up multiple router variants.

## 5 · When does routing help?

| Situation | Router helps? | Why |
|---|---|---|
| Traffic mix is genuinely varied (support inbox, chatbot) | Yes | Cheap prompts stop paying frontier-model prices |
| Traffic is uniformly simple or uniformly hard | Not much | There's nothing to route between — just pick one model directly |
| You need a guaranteed model for compliance/consistency reasons | No | Call that model directly; routing adds a decision you don't want made for you |


## 🧭 Summary — so, does one endpoint really pick different models?

Yes — you saw `served_model` shift across categories from a single `model-router` deployment, with the cost/latency numbers to back it up.

### Try it yourself
- Add your own prompt to a category and re-run — does it route where you'd expect?
- Try a prompt that straddles two categories (e.g., a comparison question with a hidden complex trade-off) and see where it lands.

### Next
➡️ This is the last of the optional deep-dive labs. Head back to the [workshop README](../../README.md) for what's next, or revisit any lab above.
